# 04 — Production Pipeline

Documents the tuned XGBoost production path, fits the full pipeline via `src.train`, inspects feature importances, and generates Kaggle-style predictions.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

%matplotlib inline

import json

import joblib
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from src.config import load_config, resolve_path
from src.data import load_test_data, load_train_data
from src.evaluation import get_feature_importances
from src.models import get_randomized_search_param_grid
from src.preprocessing import prepare_xy
from src.train import train

## Hyperparameter search (research narrative)

The original Colab workflow used `RandomizedSearchCV` (50 iterations, 5-fold CV, `roc_auc` scoring) over the XGBoost + `SelectFromModel` pipeline. Best CV ROC-AUC: **0.916**. Parameters are frozen in `config.yaml` for reproducible training.

In [ ]:
config = load_config()
param_grid = get_randomized_search_param_grid()
best_params = config["xgb_best_params"]

print("Search space (excerpt):")
for k, v in list(param_grid.items())[:4]:
    print(f"  {k}: {v}")
print("...")
print("\nFrozen best parameters:")
pd.Series(best_params, name="value").to_frame()

## Train production artifact

Equivalent to `python -m src.train --config config.yaml`. Fits on the **full** training set after reporting holdout metrics.

In [ ]:
result = train()
metrics = result["holdout_metrics"]
print("Holdout metrics:")
for k, v in metrics.items():
    print(f"  {k}: {v:.4f}")
print(f"\nModel saved to: {resolve_path(config, 'model')}")

## Feature importance (top 20)

In [ ]:
pipeline = joblib.load(resolve_path(config, "model"))
importance_df = get_feature_importances(pipeline, top_n=20)
display(importance_df)

plt.figure(figsize=(10, 6))
sns.barplot(data=importance_df, y="feature", x="importance", palette="viridis")
plt.title("Top 20 features — production XGBoost")
plt.tight_layout()
plt.show()

## Generate test predictions

In [ ]:
df_test = load_test_data()
X_test, _ = prepare_xy(df_test)
proba = pipeline.predict_proba(X_test)[:, 1]

submission = pd.DataFrame({"id": df_test["id"], "Churn": proba})
pred_path = resolve_path(config, "predictions")
pred_path.parent.mkdir(parents=True, exist_ok=True)
submission.to_csv(pred_path, index=False)
print(f"Saved {len(submission):,} predictions to {pred_path}")
submission.head()

## Next steps

- Serve the serialized pipeline via `streamlit run app/streamlit_app.py`
- Metrics JSON at `outputs/metrics.json` powers README badges